# Human-Vehicle Interactions, Run Locally

Runs **every** annotated clip through a local Qwen3.5 model, twice:

- **whole clip** — one call per clip, the model sees all of it at once;
- **8 s / 4 s windows** — one call per window, neighbours overlapping by four seconds.

Nothing leaves the machine. No API key, no upload, no per-token cost. What it costs instead is
time: a local call takes minutes, and the windowed pass makes roughly one call per four seconds of
footage.

The clips come from `notebooks/outputs/annotated/`, which is what
`human_vehicle_tracking.ipynb` fills. Everything else lives in `human_vehicle.interactions`,
`human_vehicle.vlm` and `human_vehicle.merge`; this notebook picks a model and runs the loop.

## Why the two modes are not directly comparable

The whole-clip pass reports each event once. The windowed pass reports it once **per window that
saw it**, on purpose — the second sighting is corroboration, and `find_interactions`
de-duplicates nothing. Section 6 collapses them with `merge_interactions`, and that merged count is
the one to read against the whole-clip pass.

Windowing generally finds more, because a model given four seconds of context attends to it more
closely than one given thirty. It also costs about seven times as many calls.

## How a window reaches a local model

Neither local runtime accepts a time range, so `QwenBackend` cuts the window out with ffmpeg and
shows the model that segment as a clip in its own right, beginning at zero. It then puts the
reported times back on the full clip's clock itself.

That is why there is no live timebase check here, unlike the Gemini notebook: the model is never
asked to add an offset, so there is no instruction it could disobey.

## 1. Setup

In [ ]:
import platform
import time
from pathlib import Path
from typing import Any

import cv2

from human_vehicle.interactions import (
    InteractionRun,
    find_interactions,
    make_windows,
    probe_duration,
    run_slug,
    run_tag,
)
from human_vehicle.merge import merge_interactions, merge_summary
from human_vehicle.vlm import QwenBackend


def find_repo_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root (no pyproject.toml above the cwd).")


REPO_ROOT = find_repo_root()
CLIPS_DIR = REPO_ROOT / "notebooks" / "outputs" / "annotated"
OUTPUT_DIR = REPO_ROOT / "notebooks" / "outputs" / "interactions"
CLIPS_DIR.mkdir(parents=True, exist_ok=True)

# The model id has to match the runtime this platform installed -- MLX needs a converted build,
# transformers the original weights -- so it is chosen the same way the backend chooses its runtime.
# Swap in a larger one for quality: on Apple Silicon "mlx-community/Qwen3.5-27B-4bit" is about 14 GB
# and fits 32 GB comfortably; 9B is the faster first run.
APPLE_SILICON = platform.system() == "Darwin" and platform.machine() == "arm64"
MODEL_ID = "mlx-community/Qwen3.5-9B-4bit" if APPLE_SILICON else "Qwen/Qwen3.5-9B"

WINDOW_S = 8.0
STRIDE_S = 4.0

# Weights are not loaded until the first call, so this line is free and costs no disk.
BACKEND = QwenBackend(MODEL_ID)

print(f"repo root:   {REPO_ROOT}")
print(f"clips in:    {CLIPS_DIR}")
print(f"records out: {OUTPUT_DIR}")
print(f"model:       {MODEL_ID}")
print(f"runtime:     {BACKEND.config['runtime']}  (chosen by platform, not by what imports)")
print(f"windows:     {WINDOW_S:g}s every {STRIDE_S:g}s, and one whole-clip pass")
print(f"tolerance:   {BACKEND.tolerance_s:g}s at a window edge")

## 2. The clips

Whatever is in `notebooks/outputs/annotated/`. Put the annotated renders there — the mp4s
`render_tracked_video` writes, not the source footage. An un-annotated clip runs perfectly happily
and answers a different question, with every id list empty.

The call estimate below is what the windowed pass will cost. Multiply by however long one call takes
on your machine to decide whether to run it now or leave it going.

In [ ]:
def probe(path: Path) -> dict[str, Any]:
    capture = cv2.VideoCapture(str(path))
    if not capture.isOpened():
        raise RuntimeError(f"could not open {path}")
    info = {
        "path": path,
        "clip_id": path.stem,
        "width": int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "duration_s": probe_duration(path),
    }
    capture.release()
    return info


CLIPS = [probe(path) for path in sorted(CLIPS_DIR.glob("*.mp4"))]
if not CLIPS:
    raise RuntimeError(f"no mp4 files in {CLIPS_DIR} - render some with human_vehicle_tracking.ipynb first")

_calls = sum(len(make_windows(round(info["duration_s"], 1), WINDOW_S, STRIDE_S)) for info in CLIPS)
for info in CLIPS:
    size = f"{info['width']}x{info['height']}"
    windows = len(make_windows(round(info["duration_s"], 1), WINDOW_S, STRIDE_S))
    print(f"  {info['clip_id'][:44]:<44} {info['duration_s']:6.1f}s {size:>11} {windows:>3} window(s)")

SMOKE = min(CLIPS, key=lambda info: info["duration_s"])
print(f"\n{len(CLIPS)} clip(s): {len(CLIPS)} whole-clip call(s) + about {_calls} windowed call(s)")
print(f"smoke clip:  {SMOKE['clip_id']}  ({SMOKE['duration_s']:.1f}s)")

## 3. Smoke check

One whole-clip call on the shortest clip, before committing to the full sweep.

**This is where the weights load**, so expect a long first run — several gigabytes downloaded on the
very first use, then a minute or two of loading on every later kernel. It is also where a model id
that does not exist, or does not fit in memory, fails: far better here than forty calls into a
sweep.

It asserts only that the call came back without error. Whether the answer is any good is section 5's
business.

In [ ]:
_started = time.perf_counter()
_smoke = find_interactions(SMOKE["path"], BACKEND, clip_id=SMOKE["clip_id"])
# `run.error` only says how many calls failed; the reason is on the call itself, so surface that --
# otherwise the first thing that goes wrong here is an assertion with nothing to act on.
if _smoke.error is not None:
    raise AssertionError(f"smoke call failed: {_smoke.windows[0].error or _smoke.error}")

print(f"  clip:    {SMOKE['clip_id']} ({_smoke.duration_s:.1f}s)")
print(f"  json:    {len(_smoke.interactions)} interaction(s), {len(_smoke.malformed)} malformed")
print(f"  device:  {BACKEND.config['device']}  |  runtime {BACKEND.config['runtime_version']}")
print(f"  elapsed: {time.perf_counter() - _started:.1f}s, including the model load")
for _item in _smoke.interactions:
    print(
        f"    {_item.start_time_s:5.1f}-{_item.end_time_s:5.1f}s  "
        f"{_item.person_ids} / {_item.vehicle_ids}  {_item.interaction[:44]}"
    )
print("\nsmoke check: PASS")

## 4. Running a clip

`run_clip` writes the record to `outputs/interactions/<clip>/<run_tag>.json`. `run_tag` carries the
model, the windowing and the start time, so the two modes land side by side and a repeat never
overwrites an earlier run.

A windowed run also gets a `__merged.json` beside it. The record is written first: the calls took
minutes, and the merged file is derived and regenerable from it.

In [ ]:
def run_clip(info: dict[str, Any], *, window_s: float | None = None, stride_s: float | None = None) -> InteractionRun:
    """One clip through the local model, with the record written to OUTPUT_DIR."""
    run = find_interactions(info["path"], BACKEND, window_s=window_s, stride_s=stride_s, clip_id=info["clip_id"])
    note = run.error or (
        f"{len(run.interactions)} interaction(s), {len(run.malformed)} malformed"
        + (f", {run.failed_windows} call(s) FAILED" if run.failed_windows else "")
    )
    print(f"{run.clip_id[:44]}: {note} in {run.elapsed_s:.1f}s")

    path = OUTPUT_DIR / run.clip_id / f"{run_tag(run)}.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(run.model_dump_json(indent=2), encoding="utf-8")

    # Only once the record is safely on disk: merging is derived work and can raise on a record
    # naming a window that does not exist, which must never cost the record itself.
    if run.windowed:
        merged = merge_interactions(run)
        merged_path = path.with_name(f"{run_tag(run)}__merged.json")
        merged_path.write_text(merged.model_dump_json(indent=2), encoding="utf-8")
        print(f"  {merge_summary(merged)}")
    return run


def run_all(*, window_s: float | None = None, stride_s: float | None = None) -> list[InteractionRun]:
    """Every clip, in folder order. Returns the runs; failures are recorded, not raised."""
    label = "whole clip" if window_s is None else f"{window_s:g}s/{stride_s:g}s windows"
    print(f"--- {len(CLIPS)} clip(s), {label} ---")
    started = time.perf_counter()
    runs = [run_clip(info, window_s=window_s, stride_s=stride_s) for info in CLIPS]
    print(f"--- done in {(time.perf_counter() - started) / 60:.1f} min ---\n")
    return runs


def summarize(runs: list[InteractionRun], title: str) -> None:
    header = f"{title:<26} {'calls':>5} {'fail':>5} {'found':>6} {'mal':>4} {'sec':>7}"
    print(header)
    print("-" * len(header))
    for run in runs:
        print(
            f"{run.clip_id[:26]:<26} {len(run.windows):>5} {run.failed_windows:>5} "
            f"{len(run.interactions):>6} {len(run.malformed):>4} {run.elapsed_s:>7.1f}"
        )
    print("-" * len(header))
    print(
        f"{f'{len(runs)} clips':<26} {sum(len(r.windows) for r in runs):>5} "
        f"{sum(r.failed_windows for r in runs):>5} {sum(len(r.interactions) for r in runs):>6} "
        f"{sum(len(r.malformed) for r in runs):>4} {sum(r.elapsed_s for r in runs):>7.1f}"
    )

## 5. Mode A — the whole clip

One call per clip. No duplicates and nothing to stitch together.

Watch `malformed` and the failure count. A long clip can exhaust `max_new_tokens` before the JSON is
closed, which arrives as an unparseable answer rather than an error — if that happens, either raise
the budget with `QwenBackend(MODEL_ID, max_new_tokens=32768)` or use the windowed mode below, where
each answer is shorter.

In [ ]:
runs_whole = run_all()
summarize(runs_whole, "whole clip")

## 6. Mode B — 8 s windows every 4 s

Each window is cut to its own short mp4 and shown as a clip in its own right; the times come back
corrected onto the full clip's clock.

The overlap is four seconds, so any event shorter than that is seen whole by at least one window —
which matters for direction, since "entering" and "exiting" look alike frame by frame and a
truncated view of either is easy to call wrong.

`found` counts **sightings**, not events. The merged line under each clip is the event count, and
that is what compares with mode A.

In [ ]:
runs_windowed = run_all(window_s=WINDOW_S, stride_s=STRIDE_S)
summarize(runs_windowed, f"{WINDOW_S:g}s/{STRIDE_S:g}s windows")

## 7. The two modes side by side

`whole` and `merged` are the comparable pair: one event per row each. `sightings` shows how much the
windowing duplicated, and `x` how many calls the windowed pass spent to get there.

A merged count well above the whole-clip count means windowing found things the single long call
missed, which is the usual result. A merged count *below* it is worth looking at by eye — either the
whole-clip pass invented something, or an event was split across windows and could not be merged
because the tracker never labelled one of its objects.

In [ ]:
header = f"{'clip':<30} {'whole':>6} {'sightings':>10} {'merged':>7} {'x':>4}"
print(header)
print("-" * len(header))
for _whole, _windowed in zip(runs_whole, runs_windowed, strict=True):
    _merged = merge_interactions(_windowed)
    print(
        f"{_whole.clip_id[:30]:<30} {len(_whole.interactions):>6} {len(_windowed.interactions):>10} "
        f"{_merged.merged_interaction_count:>7} {len(_windowed.windows):>4}"
    )

print(f"\nrecords in {OUTPUT_DIR.relative_to(REPO_ROOT)}/<clip>/")
print(f"  whole clip: *__{run_slug(runs_whole[0])}__*.json")
print(f"  windowed:   *__{run_slug(runs_windowed[0])}__*.json  (plus __merged.json)")

## 8. What the windowed pass merged

Only the events that more than one window saw. A high `x` with a span far longer than the
interaction it describes is the one failure mode worth knowing about: merging is transitive, so a
single over-broad span can bridge two separate episodes for the same person and vehicle into one
record.

In [ ]:
for _run in runs_windowed:
    _merged = merge_interactions(_run)
    print(merge_summary(_merged))
    for _event in _merged.interactions:
        if _event.sightings > 1:
            print(
                f"    {_event.start_time_s:5.1f}-{_event.end_time_s:5.1f}s  x{_event.sightings} "
                f"windows {_event.window_indexes}  p={_event.person_ids} "
                f"v={_event.vehicle_ids}  {_event.interaction[:40]}"
            )